# Micro-Expression Spotting & Classification (CAS(ME)^2)

In [1]:
import os
import sys

def _find_project_root(marker="pyproject.toml", max_up=8):
    path = os.path.abspath(os.getcwd())
    for _ in range(max_up):
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return os.path.abspath(os.getcwd())

project_root = _find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import gc
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from torch import nn, optim

from article.src import FrameSequence, MELabel, RealTimeProfiler
from src.apex.modules.apex_phase_spotter_roi import ApexPhaseSpotterROI
from src.dataset.modules.behavioral_features import BehavioralFeatures
from src.models.modules.spatio_temporal.spatio_temporal_cnn import SpatioTemporalCNN

In [2]:
CASME2_DIR = '/home/inadio/datasets/secondaries/cas(me)^2'
ANNOTATIONS_PATH = os.path.join(CASME2_DIR, 'CAS(ME)^2code_final.xlsx')
CACHE_DIR = os.path.join(CASME2_DIR, 'cache')

FPS = 30
MAX_SEQUENCE_LENGTH = 100
CNN_MAX_LEN = 64
BATCH_SIZE = 8
CNN_EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def compute_iou(interval_a, interval_b):
    # Matches Fang et al. 2023 (RMES) convention: interval measure is
    # (end - start), not inclusive frame count (end - start + 1).
    s_on, s_off = interval_a
    g_on, g_off = interval_b
    intersection = max(0, min(s_off, g_off) - max(s_on, g_on))
    union = (s_off - s_on) + (g_off - g_on) - intersection
    return float(intersection / union) if union > 0 else 0.0


In [4]:
rule1 = pd.read_excel(ANNOTATIONS_PATH, sheet_name='naming rule1', header=None)
sub_map = {int(row[2]): str(row[1]) for _, row in rule1.iterrows()}

rule2 = pd.read_excel(ANNOTATIONS_PATH, sheet_name='naming rule2', header=None)
stimulus_map = {str(row[1]): f"{int(row[0]):04d}" for _, row in rule2.iterrows()}

df = pd.read_excel(ANNOTATIONS_PATH, sheet_name='CASFEcode_final', header=None)
df.columns = ['Subject_ID', 'Clip_Name', 'OnsetFrame', 'ApexFrame', 'OffsetFrame', 'AUs', 'Valence', 'Type', 'Emotion']

df = df[df['Type'].eq('micro-expression')].copy()
df = df[(df['OffsetFrame'] - df['OnsetFrame'] + 1).le(MAX_SEQUENCE_LENGTH)].copy()

In [5]:
spotter = ApexPhaseSpotterROI(cutoff_ratio=0.30, show_frame=False, fps=FPS)
extractor = BehavioralFeatures()
profiler = RealTimeProfiler()

all_features = []
npz_paths = []
feat_intervals = []
spot_intervals = []
gt_intervals = []
labels = []
groups = []
frame_counts = []
clipped_sequences = []

W0000 00:00:1788744983.051417   87227 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788744983.056932   88175 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788744983.068903   88188 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
# Multiple ME rows can share the same source clip (e.g. several onset/offset
# events annotated within one video) - group by resolved npz path so each
# source clip is decompressed from disk exactly once, instead of once per row.
rows_by_npz_path = {}
for _, row in df.iterrows():
    subject_id = int(row['Subject_ID'])
    clip_name = str(row['Clip_Name'])

    subject_prefix = sub_map.get(subject_id)
    stimulus_code = stimulus_map.get(clip_name.split('_')[0])

    if subject_prefix is None or stimulus_code is None:
        continue

    npz_path = os.path.join(CACHE_DIR, f"{subject_prefix}_{stimulus_code}.npz")

    if not os.path.exists(npz_path):
        continue

    rows_by_npz_path.setdefault(npz_path, []).append((subject_prefix, row))

for npz_path, rows in rows_by_npz_path.items():
    full_seq = FrameSequence.from_npz(npz_path, fps=FPS)
    if full_seq is None or len(full_seq) == 0:
        continue

    magnitudes = full_seq.magnitudes
    seq_len = len(magnitudes)

    try:
        _, phases_dict = spotter._find_apex_phase(magnitudes, phase_mode='onset_apex_offset')
    except Exception:
        phases_dict = {}

    for subject_prefix, row in rows:
        emotion_raw = row['Emotion']

        label = MELabel.map(emotion_raw, target='2-class')
        if label is None:
            continue

        gt_onset = int(row['OnsetFrame'])
        gt_offset = int(row['OffsetFrame'])

        best_iou = -1
        best_phase = None
        best_onset_s = None
        best_offset_s = None

        for apex_idx, phase in phases_dict.items():
            onset_s = phase['start']
            offset_s = phase['end']

            iou = compute_iou((onset_s, offset_s), (gt_onset, gt_offset))
            if iou > best_iou:
                best_iou = iou
                best_phase = phase
                best_onset_s = onset_s
                best_offset_s = offset_s

        if best_iou > 0:
            feat_onset = best_phase['start']
            feat_offset = best_phase['end']
            spot_onset = best_onset_s
            spot_offset = best_offset_s
        else:
            feat_onset = gt_onset
            feat_offset = gt_offset
            spot_onset = gt_onset
            spot_offset = gt_offset

        clipped = full_seq.clip(feat_onset, feat_offset)
        if clipped is None or len(clipped) == 0:
            continue

        features = extractor._extract(torch.from_numpy(clipped.flow)).cpu().numpy()

        all_features.append(features)
        npz_paths.append(npz_path)
        feat_intervals.append((feat_onset, feat_offset))
        spot_intervals.append((spot_onset, spot_offset))
        gt_intervals.append((gt_onset, gt_offset))
        labels.append(label)
        groups.append(subject_prefix)
        frame_counts.append(seq_len)
        clipped_sequences.append(clipped)

        del features

    del full_seq
    gc.collect()

In [7]:
print(f"Filtered Micro-Expressions: {len(df)}")
print(f"Valid Sequences: {len(npz_paths)}")
print(f"Subjects: {len(set(groups))}")

Filtered Micro-Expressions: 57
Valid Sequences: 44
Subjects: 14


In [8]:
ious = []
tps = 0
for s, g in zip(spot_intervals, gt_intervals):
    iou = compute_iou(s, g)
    ious.append(iou)
    if iou >= 0.5:
        tps += 1

n_samples = len(spot_intervals)
spot_prec = tps / n_samples if n_samples > 0 else 0.0
spot_rec = tps / n_samples if n_samples > 0 else 0.0
spot_f1 = 2 * spot_prec * spot_rec / (spot_prec + spot_rec) if (spot_prec + spot_rec) > 0 else 0.0

print("=== Spotting Performance : CAS(ME)^2 ===")
print(f"Total Samples:       {n_samples}")
print(f"True Positives:      {tps} (IoU >= 0.5)")
print(f"Average IoU:         {np.mean(ious):.4f}")
print(f"Spotting Precision:  {spot_prec:.4f}")
print(f"Spotting Recall:     {spot_rec:.4f}")
print(f"Spotting F1-Score:   {spot_f1:.4f}")

=== Spotting Performance : CAS(ME)^2 ===
Total Samples:       44
True Positives:      22 (IoU >= 0.5)
Average IoU:         0.4934
Spotting Precision:  0.5000
Spotting Recall:     0.5000
Spotting F1-Score:   0.5000


In [9]:
y, class_names = MELabel.encode(labels)
groups_arr = np.asarray(groups)

X = np.stack([np.concatenate([features.mean(axis=0), features.std(axis=0)]) for features in all_features])

del all_features
gc.collect()

splits = list(LeaveOneGroupOut().split(X, y, groups_arr))

print(f"LOSO Cross-Validation")
print(f"Subjects: {len(np.unique(groups_arr))}")
print(f"Splits: {len(splits)}")
print(f"Samples: {len(X)}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(class_names)}")

LOSO Cross-Validation
Subjects: 14
Splits: 14
Samples: 44
Features: 94
Classes: 2


In [10]:
svm_preds = np.zeros_like(y)

for train_idx, test_idx in splits:
    scaler = RobustScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])

    clf = SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced', random_state=42)
    clf.fit(X_train, y[train_idx])
    svm_preds[test_idx] = clf.predict(X_test)

print("=== SVM Classification : LOSO ===")
print(f"Accuracy:        {accuracy_score(y, svm_preds):.4f}")
print(f"Macro F1-Score:  {f1_score(y, svm_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Precision: {precision_score(y, svm_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y, svm_preds, average='macro', zero_division=0):.4f}")

=== SVM Classification : LOSO ===
Accuracy:        0.5455
Macro F1-Score:  0.4762
Macro Precision: 0.4764
Macro Recall:    0.4782


In [11]:
cnn_preds = np.zeros_like(y)

for train_idx, test_idx in splits:
    train_seqs = [clipped_sequences[i] for i in train_idx]
    test_seqs = [clipped_sequences[i] for i in test_idx]

    y_train = torch.tensor(y[train_idx], dtype=torch.long, device=DEVICE)

    model = SpatioTemporalCNN(in_channels=10, num_classes=len(class_names), dropout_p=0.3).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for _ in range(CNN_EPOCHS):
        permutation = torch.randperm(len(train_seqs))
        for start in range(0, len(train_seqs), BATCH_SIZE):
            indices = permutation[start : start + BATCH_SIZE]
            if len(indices) < 2:
                continue
            batch_x = FrameSequence.pad_batch([train_seqs[i] for i in indices], max_len=CNN_MAX_LEN, device=DEVICE)
            batch_y = y_train[indices]
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(test_seqs), BATCH_SIZE):
            batch = test_seqs[start : start + BATCH_SIZE]
            batch_x = FrameSequence.pad_batch(batch, max_len=CNN_MAX_LEN, device=DEVICE)
            predictions.append(torch.argmax(model(batch_x), dim=1).cpu().numpy())

    if predictions:
        cnn_preds[test_idx] = np.concatenate(predictions)

    del train_seqs, test_seqs, model, optimizer, criterion
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("=== SpatioTemporalCNN Classification : LOSO ===")
print(f"Accuracy:        {accuracy_score(y, cnn_preds):.4f}")
print(f"Macro F1-Score:  {f1_score(y, cnn_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Precision: {precision_score(y, cnn_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y, cnn_preds, average='macro', zero_division=0):.4f}")

=== SpatioTemporalCNN Classification : LOSO ===
Accuracy:        0.6818
Macro F1-Score:  0.6562
Macro Precision: 0.6536
Macro Recall:    0.6621


In [12]:
print("Classification Report:")
print(classification_report(y, cnn_preds, target_names=class_names, zero_division=0))

Classification Report:
              precision    recall  f1-score   support

    negative       0.78      0.72      0.75        29
    positive       0.53      0.60      0.56        15

    accuracy                           0.68        44
   macro avg       0.65      0.66      0.66        44
weighted avg       0.69      0.68      0.69        44



In [13]:
cnn_full = SpatioTemporalCNN(in_channels=10, num_classes=len(class_names)).to(DEVICE)
cnn_full.eval()

for i in range(len(npz_paths)):
    with profiler.record('spot'):
        magnitudes_tmp = np.load(npz_paths[i])['magnitudes'].tolist()
        candidates = spotter._find_apex_phase(magnitudes_tmp, phase_mode='onset_apex_offset')

    clipped = clipped_sequences[i]
    if clipped is None or len(clipped) == 0:
        continue

    with profiler.record('cnn_infer'):
        x_cnn = clipped.to_tensor(device=DEVICE)
        with torch.no_grad():
            cnn_full(x_cnn)

    del x_cnn
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=== Real-Time Performance ===")
print(profiler.report(frame_counts).to_string(float_format='%.4f'))

=== Real-Time Performance ===
                    Metric             Value
0            Latency: Spot    3.310 ms / seq
1       Latency: Cnn_infer    1.629 ms / seq
2   Total Sequence Latency          4.939 ms
3  Average Sequence Length     2466.1 frames
4  Estimated Frame Latency  0.002 ms / frame
5         Throughput (FPS)      499332.0 FPS


In [14]:
print("=== Final Evaluation Summary ===")
summary = pd.DataFrame([
    {'Model': 'SVM', 'Accuracy': accuracy_score(y, svm_preds), 'Macro F1': f1_score(y, svm_preds, average='macro', zero_division=0)},
    {'Model': 'SpatioTemporalCNN', 'Accuracy': accuracy_score(y, cnn_preds), 'Macro F1': f1_score(y, cnn_preds, average='macro', zero_division=0)},
])
print(summary.to_string(index=False, float_format='%.4f'))

del extractor, profiler, spotter, cnn_full
gc.collect()

=== Final Evaluation Summary ===
            Model  Accuracy  Macro F1
              SVM    0.5455    0.4762
SpatioTemporalCNN    0.6818    0.6562


143